# Bab 6: Pohon Keputusan dan Pohon Regresi

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab06/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab06")
print("Folder kerja:", os.getcwd())

## Listing 6.1: ID3 yang ditulis sendiri

`kode/bab06/id3_numpy.py`

In [ ]:
import numpy as np
import pandas as pd

data = pd.DataFrame({
    "kadar_air": ["rendah"] * 4 + ["sedang"] * 4 + ["tinggi"] * 4,
    "cacat": ["sedikit", "sedikit", "sedikit", "banyak",
              "sedikit", "sedikit", "banyak", "banyak",
              "sedikit", "sedikit", "banyak", "banyak"],
    "proses": ["GB", "FW", "FW", "GB", "GB", "FW",
               "GB", "FW", "FW", "GB", "GB", "FW"],
    "aroma": ["kuat", "kuat", "lemah", "kuat", "kuat", "lemah",
              "kuat", "lemah", "kuat", "kuat", "lemah", "kuat"],
    "keputusan": ["terima"] * 3 + ["tolak"] + ["terima"] * 2
                 + ["tolak"] * 6,
})

def entropi(y):
    p = y.value_counts(normalize=True).values
    return float(-(p * np.log2(p)).sum())

def gain(df, atribut, target="keputusan"):
    bobot = df[atribut].value_counts(normalize=True)
    sisa = sum(bobot[v] * entropi(df.loc[df[atribut] == v, target])
               for v in bobot.index)
    return entropi(df[target]) - sisa

def id3(df, atribut, kedalaman=0):
    y = df["keputusan"]
    if y.nunique() == 1 or not atribut:
        print("    " * kedalaman + f"-> {y.mode()[0]} ({len(y)})")
        return
    g = {a: gain(df, a) for a in atribut}
    terbaik = max(g, key=g.get)
    print("    " * kedalaman + "gain: " +
          ", ".join(f"{a}={v:.4f}" for a, v in g.items()))
    for v, bagian in df.groupby(terbaik):
        print("    " * kedalaman + f"[{terbaik} = {v}]")
        id3(bagian, [a for a in atribut if a != terbaik], kedalaman + 1)

print(f"H(S) = {entropi(data['keputusan']):.4f}")
id3(data, ["kadar_air", "cacat", "proses", "aroma"])

## Listing 6.2: Mencari ambang terbaik

`kode/bab06/ambang.py`

In [ ]:
import numpy as np

kadar = np.array([11.2, 11.8, 12.1, 12.4, 12.6, 13.0, 13.5, 14.1])
label = np.array([0, 0, 0, 1, 0, 1, 1, 1])        # 1 = tolak

def gini(y):
    if len(y) == 0:
        return 0.0
    p = y.mean()
    return 1 - p ** 2 - (1 - p) ** 2

calon = (kadar[:-1] + kadar[1:]) / 2
for t in calon:
    kiri, kanan = label[kadar <= t], label[kadar > t]
    g = (len(kiri) * gini(kiri) + len(kanan) * gini(kanan)) / len(label)
    print(f"ambang {t:5.2f}: Gini terbobot = {g:.3f}")

## Listing 6.3: Pohon keputusan untuk kiriman biji

`kode/bab06/pohon_biji.py`

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv")
fitur = ["kadar_air", "cacat_primer", "cacat_sekunder",
         "ukuran_screen", "densitas_gl", "skor_total"]
X, y = biji[fitur], biji["keputusan"]

pohon = DecisionTreeClassifier(criterion="entropy",
                               max_depth=3, random_state=0)
pohon.fit(X, y)
print(export_text(pohon, feature_names=fitur,
                  show_weights=True, decimals=1))
print("akurasi latih:", round(pohon.score(X, y), 3))

## Listing 6.4: Akurasi menurut kedalaman dan pemangkasan biaya–kompleksitas

`kode/bab06/pangkas.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv")
fitur = ["kadar_air", "cacat_primer", "cacat_sekunder",
         "ukuran_screen", "densitas_gl", "skor_total",
         "ketinggian_mdpl", "lama_simpan_hari"]
Xl, Xu, yl, yu = train_test_split(biji[fitur], biji["keputusan"],
                                  test_size=.3, random_state=0,
                                  stratify=biji["keputusan"])
baris = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, 12, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xl, yl)
    baris.append((d or 99, m.get_n_leaves(), m.score(Xl, yl),
                  m.score(Xu, yu)))
tab = pd.DataFrame(baris, columns=["kedalaman", "daun", "latih", "uji"])
print(tab.round(3).to_string(index=False))
tab.to_csv("gambar/bab06/kedalaman.dat", sep=" ", index=False)

jalur = DecisionTreeClassifier(random_state=0)
alfa = jalur.cost_complexity_pruning_path(Xl, yl).ccp_alphas
alfa = np.unique(np.round(alfa, 4))[::6]
skor = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a,
        random_state=0), Xl, yl, cv=5).mean() for a in alfa]
terbaik = alfa[int(np.argmax(skor))]
m = DecisionTreeClassifier(ccp_alpha=terbaik, random_state=0).fit(Xl, yl)
print(f"alfa terbaik {terbaik:.4f}: {m.get_n_leaves()} daun, "
      f"uji {m.score(Xu, yu):.3f}")

## Listing 6.5: Pohon regresi untuk skor cupping

`kode/bab06/pohon_regresi.py`

In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv")
biji["full_washed"] = (biji["proses"] == "full washed").astype(int)
fitur = ["ketinggian_mdpl", "cacat_primer", "cacat_sekunder",
         "kadar_air", "jamur", "full_washed"]
Xl, Xu, yl, yu = train_test_split(biji[fitur], biji["skor_total"],
                                  test_size=.3, random_state=0)
print("MAE tebak rata-rata:",
      round(mean_absolute_error(yu, [yl.mean()] * len(yu)), 2))
for d in [1, 2, 3, 5, 8]:
    m = DecisionTreeRegressor(max_depth=d, min_samples_leaf=20,
                              random_state=0).fit(Xl, yl)
    print(f"kedalaman {d}: MAE uji = "
          f"{mean_absolute_error(yu, m.predict(Xu)):.2f}")
m = DecisionTreeRegressor(max_depth=2, min_samples_leaf=20).fit(Xl, yl)
print(export_text(m, feature_names=fitur, decimals=1))

## Listing 6.6: Studi kasus aturan terima–tolak

`kode/bab06/studi_kasus.py`

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text

DATA = "kode/data/"
biji = pd.read_csv(DATA + "mutu_biji.csv", parse_dates=["tanggal"])
fitur = ["kadar_air", "cacat_primer", "cacat_sekunder",
         "ukuran_screen", "densitas_gl", "skor_total"]
latih = biji[biji["tanggal"] < "2025-01-01"]
uji = biji[biji["tanggal"] >= "2025-01-01"]
print("latih:", len(latih), " uji:", len(uji))

lama = ((uji["kadar_air"] <= 12.5) & (uji["cacat_primer"] <= 5))
pred_lama = lama.map({True: "terima", False: "tolak"})
print("aturan lama, kecocokan:",
      round(accuracy_score(uji["keputusan"], pred_lama), 3))

cari = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    {"max_leaf_nodes": [4, 6, 8, 10, 12, 16, 24],
     "min_samples_leaf": [10, 20, 40]}, cv=5)
cari.fit(latih[fitur], latih["keputusan"])
print("terbaik:", cari.best_params_,
      "CV:", round(cari.best_score_, 3))
print("pohon terbaik, kecocokan uji:",
      round(cari.score(uji[fitur], uji["keputusan"]), 3))

pohon = DecisionTreeClassifier(max_leaf_nodes=8,
                               min_samples_leaf=20, random_state=0)
pohon.fit(latih[fitur], latih["keputusan"])
pred = pohon.predict(uji[fitur])
print("pohon 8 daun, kecocokan uji:",
      round(accuracy_score(uji["keputusan"], pred), 3))
print(confusion_matrix(uji["keputusan"], pred,
                       labels=["terima", "tolak"]))
print(export_text(pohon, feature_names=fitur, decimals=1))
penting = pd.Series(pohon.feature_importances_, index=fitur)
print(penting.sort_values(ascending=False).round(3).to_string())